# Test-Abruf Sonnendach-Daten (BFE) für ein Gebiet in Winterthur

Ziel: herausfinden, welche Felder die geo.admin-API pro Dachfläche liefert,
wie viele Dachflächen und Gebäude es in einem kleinen Gebiet gibt,
und ob die seitenweise Abfrage (max. 200 pro Abfrage) funktioniert.

In VS Code kann diese Datei Zelle für Zelle ausgeführt werden ("Run Cell"
über jeder # %% -Zeile) oder in ein Notebook kopiert werden.



In [15]:
import requests
import pandas as pd

URL = "https://api3.geo.admin.ch/rest/services/ech/MapServer/identify"
LAYER = "ch.bfe.solarenergie-eignung-daecher"
PAGE_SIZE = 200
MAX_OFFSET = 20_000
TIMEOUT_SECONDS = 60

# Testgebiet: ca. 500 m x 500 m um die Winterthurer Altstadt,
# Koordinaten im Schweizer System LV95 (EPSG:2056): xmin, ymin, xmax, ymax
BBOX = (2696600, 1261600, 2697100, 1262100)

session = requests.Session()


In [16]:
def abfrage(bbox, offset=0, limit=PAGE_SIZE):
    """Eine Seite Dachflächen für ein Rechteck abfragen."""
    if not 1 <= limit <= PAGE_SIZE:
        raise ValueError(f"limit muss zwischen 1 und {PAGE_SIZE} liegen.")

    params = {
        "geometry": ",".join(str(value) for value in bbox),
        "geometryType": "esriGeometryEnvelope",
        "layers": f"all:{LAYER}",
        "tolerance": 0,
        "sr": 2056,
        "returnGeometry": "false",
        "limit": limit,
        "offset": offset,
    }
    response = session.get(URL, params=params, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()
    payload = response.json()
    results = payload.get("results") if isinstance(payload, dict) else None
    if not isinstance(results, list):
        raise ValueError("Die API-Antwort enthält keine gültige 'results'-Liste.")
    return results


# Erste Seite holen und anschauen
erste_seite = abfrage(BBOX)
print("Anzahl Dachflächen auf Seite 1:", len(erste_seite))
print(
    "Schlüssel eines Resultats:",
    list(erste_seite[0].keys()) if erste_seite else "keine Resultate",
)


Anzahl Dachflächen auf Seite 1: 201
Schlüssel eines Resultats: ['layerBodId', 'layerName', 'featureId', 'id', 'attributes']


In [17]:
# Welche Attribute gibt es pro Dachfläche? (die eigentliche Frage des Test-Abrufs)
if erste_seite:
    for key, value in erste_seite[0].get("attributes", {}).items():
        print(f"{key:25} {value}")


monate                    [10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 12, 11]
building_id               398922
a_param                   [0.9395000591, 0.9835008748, 0.9842962419, 0.9965533668, 0.9998395246, 0.9905156403, 0.9823735074, 0.9670224334, 0.8850366497, 0.8371521151, 0.6960470151, 0.8492116983]
b_param                   [0.9721498762, 0.9627576649, 0.9817739393, 0.9845374167, 0.9684455873, 0.9419326359, 0.9733517239, 0.9781084652, 0.9384767674, 0.9145921726, 0.8848662852, 0.9400992085]
c_param                   [-1.2322757245, -0.1909195901, -0.9396948546, -1.7639077571, -0.3881743819, 2.1416148871, -0.5265463547, -1.0436394162, -0.1085708859, -0.4929154727, 0.3643084819, -0.9982511924]
heizgradtage              [229.0, 61.0, 3.0, 1.0, 17.0, 100.0, 241.0, 455.0, 534.0, 591.0, 580.0, 447.0]
bedarf_heizung            96036
bedarf_warmwasser         7358
datum_aenderung           10.12.2020
datum_erstellung          10.12.2020
dg_heizung                10
dg_waermebedarf           11
duschg

In [18]:
# Alle Seiten holen. Die bereits geladene erste Seite wird wiederverwendet.
alle = []
gesehen = set()
offset = 0

while offset < MAX_OFFSET:
    limit = min(PAGE_SIZE, MAX_OFFSET - offset)
    seite = erste_seite if offset == 0 else abfrage(BBOX, offset=offset, limit=limit)
    if not seite:
        break

    neue = []
    for result in seite:
        feature_id = result.get("featureId")
        if feature_id is None or feature_id not in gesehen:
            neue.append(result)
            if feature_id is not None:
                gesehen.add(feature_id)

    if not neue:
        break

    alle.extend(neue)
    print(f"offset {offset}: {len(neue)} neue Dachflächen, total {len(alle)}")

    if len(seite) < limit:
        break
    offset += limit
else:
    print(f"Sicherheits-Stopp bei offset {MAX_OFFSET}.")

df_roofs = pd.DataFrame([result.get("attributes", {}) for result in alle])
df_roofs["featureId"] = [result.get("featureId") for result in alle]
print("Dachflächen total:", len(df_roofs))
print("Spalten:", list(df_roofs.columns))


offset 0: 201 neue Dachflächen, total 201
offset 200: 200 neue Dachflächen, total 401
offset 400: 200 neue Dachflächen, total 601
offset 600: 200 neue Dachflächen, total 801
offset 800: 200 neue Dachflächen, total 1001
offset 1000: 200 neue Dachflächen, total 1201
offset 1200: 112 neue Dachflächen, total 1313
Dachflächen total: 1313
Spalten: ['monate', 'building_id', 'a_param', 'b_param', 'c_param', 'heizgradtage', 'bedarf_heizung', 'bedarf_warmwasser', 'datum_aenderung', 'datum_erstellung', 'dg_heizung', 'dg_waermebedarf', 'duschgaenge', 'flaeche_kollektoren', 'gstrahlung', 'mstrahlung', 'sb_datum_aenderung', 'sb_datum_erstellung', 'sb_objektart', 'volumen_speicher', 'waermeertrag', 'klasse', 'flaeche', 'ausrichtung', 'finanzertrag', 'stromertrag', 'stromertrag_winterhalbjahr', 'stromertrag_sommerhalbjahr', 'monats_ertrag', 'gs_serie_start', 'klasse_text', 'gwr_egid', 'df_nummer', 'neigung', 'label', 'de', 'fr', 'it', 'en', 'featureId']


In [19]:
# Wie viele Gebäude sind das? Dafür braucht es eine Gebäude-ID
# (z. B. building_id oder gwr_egid).
id_spalten = [
    column
    for column in df_roofs.columns
    if "egid" in column.lower() or "building" in column.lower()
]
print("Mögliche Gebäude-ID-Spalten:", id_spalten)
for column in id_spalten:
    print(
        f"{column}: {df_roofs[column].nunique()} verschiedene Werte, "
        f"{df_roofs[column].isna().sum()} fehlend"
    )


Mögliche Gebäude-ID-Spalten: ['building_id', 'gwr_egid']
building_id: 308 verschiedene Werte, 1 fehlend
gwr_egid: 165 verschiedene Werte, 20 fehlend


In [20]:
# Fehlende Werte und Datentypen auf einen Blick
print(df_roofs.isna().sum())
df_roofs.head()


monate                           1
building_id                      1
a_param                          1
b_param                          1
c_param                          1
heizgradtage                     1
bedarf_heizung                   1
bedarf_warmwasser                1
datum_aenderung                  1
datum_erstellung                 1
dg_heizung                       1
dg_waermebedarf                  1
duschgaenge                      1
flaeche_kollektoren              1
gstrahlung                       1
mstrahlung                       1
sb_datum_aenderung               1
sb_datum_erstellung              1
sb_objektart                     1
volumen_speicher                 1
waermeertrag                     1
klasse                           1
flaeche                          1
ausrichtung                      1
finanzertrag                     1
stromertrag                      1
stromertrag_winterhalbjahr       1
stromertrag_sommerhalbjahr       1
monats_ertrag       

,monate,building_id,a_param,b_param,c_param,heizgradtage,bedarf_heizung,bedarf_warmwasser,datum_aenderung,datum_erstellung,...,klasse_text,gwr_egid,df_nummer,neigung,label,de,fr,it,en,featureId
0,"[10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 12, 11]",398922.0,"[0.9395, 0.983501, 0.984296, 0.996553, 0.99984...","[0.97215, 0.962758, 0.981774, 0.984537, 0.9684...","[-1.232276, -0.19092, -0.939695, -1.763908, -0...","[229.0, 61.0, 3.0, 1.0, 17.0, 100.0, 241.0, 45...",96036.0,7358.0,10.12.2020,10.12.2020,...,Gut##Bonne##Buona##Good##Gut,1150418.0,1.0,2.0,16505790,NaN,NaN,NaN,NaN,16505790
1,"[10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 12, 11]",444661.0,"[0.691713, 0.81626, 0.917644, 0.980683, 0.9826...","[1.020416, 0.771665, 0.947635, 0.95291, 0.9331...","[-9.230407, 5.450854, -2.693025, -4.444327, -1...","[228.0, 60.0, 3.0, 1.0, 15.0, 98.0, 239.0, 455...",0.0,0.0,10.12.2020,10.12.2020,...,Gut##Bonne##Buona##Good##Gut,NaN,1.0,0.0,16505611,NaN,NaN,NaN,NaN,16505611
2,"[10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 12, 11]",1185237.0,"[-0.07979, 0.26874, 0.504202, 0.71904, 0.74308...","[0.594754, 0.501999, 0.616155, 0.796887, 0.753...","[4.784597, 8.074719, 8.876797, -5.612522, -0.4...","[229.0, 60.0, 3.0, 1.0, 16.0, 99.0, 240.0, 455...",343407.0,15900.0,09.12.2020,09.12.2020,...,Gering##Faible##Bassa##Low##Gering,1150015.0,1.0,0.0,16500854,NaN,NaN,NaN,NaN,16500854
3,"[10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 12, 11]",3295719.0,"[0.59599, 0.679536, 0.72901, 0.749088, 0.69131...","[1.121162, 0.854399, 0.683032, 0.906159, 0.697...","[-9.595239, 0.301591, 13.244424, 1.238019, 23....","[230.0, 62.0, 4.0, 1.0, 17.0, 101.0, 241.0, 45...",94008.0,8680.0,10.12.2020,10.12.2020,...,Mittel##Moyenne##Media##Mean##Mittel,1150410.0,1.0,34.0,16505871,NaN,NaN,NaN,NaN,16505871
4,"[10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 12, 11]",3295719.0,"[1.269918, 1.200748, 1.12421, 1.084696, 1.1705...","[0.496971, 0.979982, 1.056802, 0.945814, 1.177...","[14.786505, -5.645001, -13.71919, -10.027721, ...","[230.0, 62.0, 4.0, 1.0, 17.0, 101.0, 241.0, 45...",94008.0,8680.0,10.12.2020,10.12.2020,...,Sehr gut##Trés bonne##Molto buona##Very good##...,1150410.0,2.0,32.0,16505872,NaN,NaN,NaN,NaN,16505872


In [21]:
# Speichern für die weitere Arbeit
output_path = "solar_test_winterthur_altstadt.csv"
df_roofs.to_csv(output_path, index=False)
print(f"Gespeichert: {output_path}")


Gespeichert: solar_test_winterthur_altstadt.csv
